# **Phase de Buisness Understanding**

**L’entreprise a perdu 53% de ses clients ces 4 dernières années**

**Cela constitue une perte de revenu majeure**

### **POURQUOI ?** A moi de l’identifier et de déterminer les profils qui risquent de partir.

 
### **OBJECTIF :**  **REDUIRE CE TAUX DE PERTE EN OPTIMISANT LES COÛTS MARKETING**    
#### $\rightarrow$ connaissance des facteurs  
   
#### $\rightarrow$ ajustement de la stratégie (sanctions par les profils à risque).

### $\hookrightarrow$ Présentation du modèle à l'équipe marketing qui disposera d'un outil de ciblage des clients.
   
   


On pourrait s'interesser à la stratégie actuelle de gérer le churn de l'entreprise, comme c'est un projet et que l'on est pas sur un "vrai" contexte, supposons que la société n'a pas de solution adéquate et adopte une stratégie de ciblage général: 
Sur 1000 personnes, disons que j'appelle 500 personnes , je ne toucherai alors que 265 clients qui allait probablement churn. 
**L'objectif du modèle est de démontrer un Lift** (facteur multiplicateur) pour **passer de 265 à 400 ou 450 churners**, selon les contraintes de côut.

### **DATA** 
On dispose de 70000 observations de clients. 
En termes de données, en observations (factures), nous possédons des **données démographiques**, des **infos sur le compte** ainsi que les **services souscrits** et des **données de facturation / financières**. 

Il y a un manque de temporalité dans la donnée : **il manque la date du churn.** 
On dispose uniquement de la variable "ancienneté". On va émettre l'hypothèse uniquement dans cette partie pour avoir une modélisation réaliste du problème que par rapport à la distribution de cette variable, la perte des clients a été faite durant ces 4 dernières années (48 mois). En réalité, introduire cette info dans le dataset introduierait des hypothèses non vérifiables et potentiellement biaisantes. L'inférence est donc limitée par le manque de feature de ce dataset public. En situation réelle, l’accès aux données via SQL ou systèmes internes permettrait récuperer ces données manquantes. 



### **Modèle**

Il s’agit d’un **projet de classification binaire** :  
On cherche à déterminer si le client risque de partir ou non.

Etant donnée la pauvreté de l'inférence, on se concentrera sur des algorithmes performants.





### **CONTRAINTE METIER : ARGENT, HUMAIN.**

### Dans cette section, on va chercher à **créer une fonction de perte ajusté sur les contraintes de l'entreprise**, afin d'entrainer mon modèle sur le risque associé, plûtot que d'utiliser le risque classique (model.fit()) et de l'ajuster par rapport au Recall, ce qui engendrerais du biais. 

**Le coût de l’erreur :** 
* C’est beaucoup plus grave de prédire qu’un client **reste** alors qu’il compte **partir**. (**FAUX NEGATIF**)
$$ \text{Marge mensuelle : } 9\text{ €} $$

$$ \text{Horizon restant : } 10\text{ mois} \rightarrow 90\text{ €} $$

$$ \text{Coût de remplacement : } 105\text{ €} $$

$$ \text{Coût total d’un churner : } \approx 195\text{--}205\text{ €} $$

$$ \text{Coût fixé : } 200\text{ € par churner} $$

* Que de prédire qu’un client **part** alors qu’il comptait **rester** (un petit cadeau de la maison, une petite promotion inattendue, **FAUX POSITIF**).
$$ \text{Coût fixe par client traité : } 25\text{ €} $$

Donc on perd 200€ si FN  et 25 si FP,  **On a un début de fonction de perte**  

### Le problème revient donc à créer un modèle qui minimise le nombre de Faux négatif (FN) en prenant en compte les coûts non seulement des FN **mais également des FP** 
$$C_{FP} \, nb(FP) + C_{FN} \, nb(FN) $$
Sinon mon modèle prédit toujours que le client churn, on fais du démarchage avec une offre pour chaque client, et soit on a tout le PIB de la Chine à disposition, soit ça devient du bénévolat. 

Il y a un rapport entre l'importance du nbre de churner évité et le coût d'une erreur (FP ou FN)

En ce sens, l'ajustement du modèle, dépend de si: 
* l'entreprise cherche uniquement à réduire son taux à un certain seuil pour les stats (et dans ce cas elle doit mettre le prix)
* **ou bien l'entreprise cherche à maximiser son ROI liée au churn** (et donc trouver un équilibre entre FN et FP)

Nous sommes concernés par le second cas. ( Dans le premier, on aurait utilisé une autre fonction de perte )


On va faire une **modélisation probabiliste** du problème :  

$$
\begin{aligned}
& Y \in \{0, 1\} \text{ cible}, \quad X \in \mathbb{R}^d \text{ features} \\
& g(X) \text{ un prédicteur.} \qquad FP = \{Y=0, g(X)=1\} \\
& \phantom{g(X) \text{ un prédicteur.} \qquad} FN = \{Y=1, g(X)=0\} \\
\\
& \text{On définit } \ell_s(y, y') = 
\begin{cases} 
25 \text{ si } FP \\
200 \text{ si } FN 
\end{cases}              \text{ la fonction de perte "S"} \\
\\
& \text{Alors } \ell_s(y, y') = \mathbb{1}_{\{Y=0, g(X)=1\}} 25 + \mathbb{1}_{\{Y=1, g(X)=0\}} 200 \\
\\
& \text{Métrique : } R_s(g) = \mathbb{E}[\ell_s(g(X), Y)] = \mathbb{E} \left[ \mathbb{1}_{\{Y=0, g(X)=1\}} 25 + \mathbb{1}_{\{Y=1, g(X)=0\}} 200 \right] \\
\\
& \overset{\text{linéarité}}{=} 25 \, \mathbb{P}(\{Y=0, g(X)=1\}) + 200 \, \mathbb{P}(\{Y=1, g(X)=0\}) \\
\\
& = 25 \, \mathbb{P}(FP) + 200 \, \mathbb{P}(FN) \\
\\
& \text{et on retrouve notre formule } \text{intuitive} :  C_{FP} \, nb(FP) + C_{FN} \, nb(FN) \text{ !!} \\
\end{aligned}
$$
\\
$$ R_s(g) \text{ représente la perte moyenne en euro} \\$$
$$ \text{Le but du projet est donc de trouver un prédicteur } \hat{g} \\ 
 \text{qui minimise } \hat{R_s}(g) \text{ notre estimateur de la perte moyenne en euro !}$$

En comparant sur cette métrique l'ancienne stratégie avec la nouvelle (notre nouveau prédicteur), on pourra **estimer le gain moyen engendré par cette nouvelle stratégie** 

---
**RETOUR CONTRAINTE METIER :** Notre stratégie dépend des techniques de marketing / humaine :  
* **Promotion** $\rightarrow$ applicable à tous.  
* **Appels téléphoniques** $\rightarrow$ limites humaines, notre modèle devra trier et imaginer que j’ai dix personnes qui peuvent réaliser 50 appels par jour $\rightarrow$ 500 personnes.  
Mon modèle lui distingue 2000 personnes critiques $\rightarrow$ je devrai choisir les 500 personnes les plus à risque en choisissant ceux qui ont la plus forte probabilité de churn.  

Disons qu'on ne peut traiter que le **top 5% de nos 7000**.  
Dans l'idéal, il faudrait que sur 1000 personnes, comme ma capacité de traitement est de 5%, que parmi mon modèle affiche dans les 50 premières personnes les plus à risque, **40 personnes qui ont effectivement résilié**.  

---

 

C'est maintenant le moment de faire parler la donnée pour trouver notre modèle.